# WSJ full corpus → date + headline + body + article_id (all four eras, one compressed file)

Stays entirely inside ProQuest TDM Studio. **No export** — the combined file is far over
the 30 MB weekly cap and is meant to live here for downstream work.

Plan:
1. **Per-folder parquet** — for each of `WSJ_1889-1919`, `WSJ_1920-1964`,
   `WSJ_1965-2014`, `WSJ_2015-2026`, regex-scan every XML and stream
   `date, headline, body, article_id` to a zstd parquet (memory-safe, batched).
2. **Combine** — stitch the four per-folder parquets into one
   `wsj_full_corpus.parquet`, row-group by row-group (never loads it all into RAM).
3. **Peek** — sanity-check the combined file.

Schema:
- Date: `<NumericDate>YYYY-MM-DD</NumericDate>` (backup `<StartDate>`)
- Headline: `<TitleAtt><Title>...</Title></TitleAtt>`
- Body — **two different ProQuest products, handled by separate extractors:**
  - **1889–2014 (historical OCR database):** `<HiddenText HTMLContent="true">`, HTML-escaped
    HTML → `extract_body_hist` (double `html.unescape`, strip tags).
  - **2015–2026 (current product):** body is in `<Text>` (inside a `<TextInfo>` wrapper) →
    `extract_body_modern`. `body_extractor_for(folder)` picks the right one per folder.
- article_id: integer from `<id>.xml` (recover the file downstream as `f'{article_id}.xml'`).

Full bodies, every article — no keyword filter, no truncation, no min-length cut.

**Note on pyarrow:** this notebook builds parquet tables directly with pyarrow
(`pa.array` + explicit schema) and never uses the pandas↔arrow bridge
(`Table.from_pandas` / `.to_pandas()`), which throws extension-type errors
(`pandas.period already defined`, `arrow.py_extension_type`) on this TDM Studio
pyarrow version. pandas is used only for flexible date parsing.

## Cell 1 — shared extractors + per-folder → parquet (batched, memory-safe)

Run this once to define everything; the next cells drive it.

In [ ]:
import re, html, time, os
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The four era folders. Each is tried at the usual TDM mount points.
FOLDERS = ['WSJ_1889-1919', 'WSJ_1920-1964', 'WSJ_1965-2014', 'WSJ_2015-2026']
# 2015-2026 is a DIFFERENT ProQuest product (not the historical OCR database), so its XML
# stores the body differently and gets its own extractor — see below.
MODERN_FOLDERS = {'WSJ_2015-2026'}

def resolve(folder):
    for base in ('data', '/data', '../data', '/home/jovyan/data'):
        p = Path(base) / folder
        if p.exists():
            return p
    return None

# --- regexes on raw bytes (fast; same approach as the headlines script) ---
DATE_RE         = re.compile(rb'<NumericDate>\s*(\d{4}-\d{2}-\d{2})', re.I)
DATE_RE_BACKUP  = re.compile(rb'<StartDate>\s*(\d{4}-?\d{2}-?\d{2})', re.I)
TITLE_RE        = re.compile(rb'<TitleAtt>.*?<Title>(.*?)</Title>', re.I | re.S)
TITLE_RE_BACKUP = re.compile(rb'<Title>(.*?)</Title>', re.I | re.S)
# Body tag differs by product:
#   historical DB (1889-2014): <HiddenText> holds HTML-escaped HTML
#   current product (2015-2026): <Text> holds the body (inside a <TextInfo> wrapper)
HIDDEN_RE       = re.compile(rb'<HiddenText\b[^>]*>(.*?)</HiddenText>', re.I | re.S)
TEXT_RE         = re.compile(rb'<Text\b[^>]*>(.*?)</Text>', re.I | re.S)   # \b -> not <TextInfo>
TAG_RE          = re.compile(r'<[^>]+>')
WS_RE           = re.compile(r'\s+')
ID_RE           = re.compile(r'^(\d+)\.xml$')

def extract_date(buf):
    m = DATE_RE.search(buf) or DATE_RE_BACKUP.search(buf)
    return m.group(1).decode() if m else None

def extract_title(buf):
    m = TITLE_RE.search(buf) or TITLE_RE_BACKUP.search(buf)
    if not m:
        return ''
    raw = m.group(1).decode('utf-8', errors='replace')
    return WS_RE.sub(' ', html.unescape(raw)).strip()

def _clean(parts):
    """Decode -> unescape (twice; harmless if singly-escaped) -> strip tags -> collapse ws.
    `parts` is a list of byte-strings (all matched chunks of the body tag)."""
    if not parts:
        return ''
    s = b' '.join(parts).decode('utf-8', errors='replace')
    s = html.unescape(html.unescape(s))
    return WS_RE.sub(' ', TAG_RE.sub(' ', s)).strip()

def extract_body_hist(buf):
    """Historical DB (1889-2014): body is HTML-escaped HTML inside <HiddenText>."""
    return _clean([m.group(1) for m in HIDDEN_RE.finditer(buf) if m.group(1).strip()])

def extract_body_modern(buf):
    """Current product (2015-2026): body is in <Text> (may be several <Text> chunks)."""
    return _clean([m.group(1) for m in TEXT_RE.finditer(buf) if m.group(1).strip()])

def body_extractor_for(folder):
    return extract_body_modern if folder in MODERN_FOLDERS else extract_body_hist

# Arrow schema for every parquet we write — keeps all four files byte-compatible
SCHEMA = pa.schema([
    ('date',       pa.timestamp('ns')),
    ('headline',   pa.string()),
    ('body',       pa.string()),
    ('article_id', pa.int64()),
])

def rows_to_table(rows):
    """Build a pyarrow Table from a list of {date,headline,body,article_id} dicts,
    WITHOUT the pandas->arrow bridge. Dates parsed with pandas, then handed to arrow
    as a plain numpy datetime64[ns] array (NaT -> null)."""
    dates = pd.to_datetime([r['date'] for r in rows], errors='coerce').values
    return pa.Table.from_arrays(
        [
            pa.array(dates, type=pa.timestamp('ns')),
            pa.array([r['headline'] for r in rows], type=pa.string()),
            pa.array([r['body'] for r in rows], type=pa.string()),
            pa.array([r['article_id'] for r in rows], type=pa.int64()),
        ],
        schema=SCHEMA,
    )

# zstd level: 10 is a good speed/ratio balance on multi-GB text. Bump toward 19-22
# for smaller files at the cost of much slower writes; drop to 3 for speed.
ZSTD_LEVEL = 10
BATCH_SIZE = 20000   # rows buffered before each parquet write

def folder_to_parquet(root, out_path):
    extract_body = body_extractor_for(root.name)        # pick the era's body extractor
    files = list(root.glob('*.xml'))
    print(f'  {root.name}: {len(files):,} XML files -> {out_path.name} '
          f'[{extract_body.__name__}]', flush=True)
    writer = pq.ParquetWriter(out_path, SCHEMA, compression='zstd',
                              compression_level=ZSTD_LEVEL)
    batch, n_files, no_date, no_body, no_id = [], 0, 0, 0, 0
    t0 = time.time()
    for i, f in enumerate(files):
        try:
            buf = f.read_bytes()
        except Exception:
            continue
        d = extract_date(buf)
        t = extract_title(buf)
        b = extract_body(buf)
        m = ID_RE.match(f.name)
        aid = int(m.group(1)) if m else None
        if d is None: no_date += 1
        if not b:     no_body += 1
        if aid is None: no_id += 1
        batch.append({'date': d, 'headline': t, 'body': b, 'article_id': aid})
        n_files += 1
        if len(batch) >= BATCH_SIZE:
            writer.write_table(rows_to_table(batch)); batch = []
        if (i + 1) % 20000 == 0:
            rate = (i + 1) / (time.time() - t0)
            eta = (len(files) - (i + 1)) / rate / 60
            print(f'    {i+1:,}/{len(files):,}  rate={rate:.0f}/s  ETA={eta:.1f} min', flush=True)
    if batch:
        writer.write_table(rows_to_table(batch))
    writer.close()
    mb = os.path.getsize(out_path) / 1e6
    print(f'    done: {n_files:,} rows, {mb:,.1f} MB | '
          f'no_date={no_date:,} no_body={no_body:,} no_id={no_id:,}', flush=True)
    return {'folder': root.name, 'path': str(out_path), 'rows': n_files, 'mb': mb,
            'no_date': no_date, 'no_body': no_body, 'no_id': no_id}

print('Extractors defined. zstd level =', ZSTD_LEVEL, '| batch size =', BATCH_SIZE)


## Cell 1b — peek at REAL extracted rows before the big scan

Runs the actual extraction on the first few files of a mounted folder, builds the exact
table that gets written (same `rows_to_table` + schema as production), round-trips it
through a tiny throwaway parquet, and prints one full row back — so you can confirm the
**body is really there** before the multi-hour run. No pandas↔arrow bridge anywhere.

In [ ]:
# Set SAMPLE_FOLDER explicitly to verify a specific product, e.g. 'WSJ_2015-2026'
# to confirm the <Text> path. Defaults to the first mounted folder.
SAMPLE_FOLDER = next((f for f in FOLDERS if resolve(f)), None)
assert SAMPLE_FOLDER, 'No folder mounted.'
root = resolve(SAMPLE_FOLDER)
extract_body = body_extractor_for(SAMPLE_FOLDER)
print(f'Sampling from {root}  [{extract_body.__name__}]\n')

rows = []
for f in list(root.glob('*.xml'))[:5]:
    buf = f.read_bytes()
    m = ID_RE.match(f.name)
    rows.append({
        'date': extract_date(buf),
        'headline': extract_title(buf),
        'body': extract_body(buf),
        'article_id': int(m.group(1)) if m else None,
    })

# Same write path as production, then read straight back as pyarrow (no .to_pandas())
tmp = OUT_DIR / '_peek_sample.parquet'
pq.write_table(rows_to_table(rows), tmp, compression='zstd', compression_level=ZSTD_LEVEL)
back = pq.read_table(tmp)

print('Schema as stored in parquet:')
print(back.schema)

bodies = back.column('body').to_pylist()
print(f'\nbody length per sampled row: {[len(x) for x in bodies]}')

# Show ONE full row, body included (pure pyarrow -> python)
d   = back.column('date').to_pylist()[0]
aid = back.column('article_id').to_pylist()[0]
hl  = back.column('headline').to_pylist()[0]
bd  = bodies[0]
print('\n' + '=' * 78)
print('ONE EXAMPLE ROW (as stored in the parquet)')
print('=' * 78)
print(f'date       : {d}')
print(f'article_id : {aid}')
print(f'headline   : {hl}')
print(f'body length: {len(bd):,} chars')
print('-' * 78)
print('body:')
print(bd[:3000] + ('...' if len(bd) > 3000 else ''))

tmp.unlink()  # clean up the throwaway


## Cell 2 — build one parquet per folder

Loops over all four eras. Skips (with a warning) any folder not mounted. Each parquet is
written to `output_files/wsj_full_<folder>.parquet`.

In [ ]:
per_folder = []
for folder in FOLDERS:
    root = resolve(folder)
    if root is None:
        print(f'  [skip] {folder} not found at any mount point', flush=True)
        continue
    out_path = OUT_DIR / f'wsj_full_{folder}.parquet'
    per_folder.append(folder_to_parquet(root, out_path))

print('\nPer-folder summary:')
for r in per_folder:
    print(f"  {r['folder']:<16} rows={r['rows']:>10,}  {r['mb']:>8,.1f} MB  "
          f"no_date={r['no_date']:,}  no_body={r['no_body']:,}  no_id={r['no_id']:,}")
if not per_folder:
    print('  nothing written — no folders resolved.')


## Cell 3 — combine the per-folder parquets into one file

Streams row-group by row-group, so the full multi-GB corpus is never held in memory at
once. Folders are concatenated in chronological order (1889 → 2026); rows within a folder
keep file-scan order. For a strict global date sort, do it downstream where RAM allows.

In [ ]:
COMBINED = OUT_DIR / 'wsj_full_corpus.parquet'

# Always combine the per-folder parquets that exist ON DISK (independent of the in-memory
# `per_folder` list), so re-running after rebuilding a single folder picks up all four.
# Ordered chronologically by FOLDERS.
order = {f: i for i, f in enumerate(FOLDERS)}
paths = sorted(
    OUT_DIR.glob('wsj_full_WSJ_*.parquet'),
    key=lambda p: order.get(p.name.replace('wsj_full_', '').replace('.parquet', ''), 999)
)
print(f'Combining {len(paths)} per-folder parquets on disk (in order):')
for p in paths:
    print(f'  {p.name}')
if len(paths) < len(FOLDERS):
    print(f'  WARNING: only {len(paths)}/{len(FOLDERS)} folders present — build the missing '
          f'ones first if you want the full corpus.')

writer = pq.ParquetWriter(COMBINED, SCHEMA, compression='zstd', compression_level=ZSTD_LEVEL)
total = 0
for p in paths:
    pf = pq.ParquetFile(p)
    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(rg)
        writer.write_table(tbl)
        total += tbl.num_rows
writer.close()

mb = os.path.getsize(COMBINED) / 1e6
print(f'\nWrote {COMBINED}')
print(f'  {total:,} rows, {mb:,.1f} MB  (lives in ProQuest — not exported)')


## Cell 4 — peek at the combined file

Reads only metadata + a couple of row groups, so this stays light even though the file is
large. Pure pyarrow — no pandas↔arrow bridge.

In [ ]:
pf = pq.ParquetFile(COMBINED)
print(f'Combined file: {COMBINED.name}')
print(f'  size:       {os.path.getsize(COMBINED)/1e6:,.1f} MB')
print(f'  rows:       {pf.metadata.num_rows:,}')
print(f'  row groups: {pf.num_row_groups:,}')
print(f'  columns:    {pf.schema_arrow.names}')

def show_rows(tbl, label, k=3):
    n = tbl.num_rows
    idxs = list(range(min(k, n)))
    cols = {c: tbl.column(c).to_pylist() for c in tbl.schema.names}
    print(f'\n{label}:')
    for j in idxs:
        bd = cols['body'][j] or ''
        bd = bd[:160] + ('...' if len(bd) > 160 else '')
        print(f"  [{cols['date'][j]}] id={cols['article_id'][j]} | "
              f"{(cols['headline'][j] or '')[:80]}")
        print(f"      body: {bd}")

# First and last row groups span the earliest and latest eras
show_rows(pf.read_row_group(0), 'First rows (earliest era)')
last = pf.read_row_group(pf.num_row_groups - 1)
show_rows(last.slice(max(0, last.num_rows - 3), 3), 'Last rows (latest era)')

# Body-length stats from one mid row group (cheap; avoids full scan)
mid = pf.read_row_group(pf.num_row_groups // 2)
lens = sorted(len(x or '') for x in mid.column('body').to_pylist())
n = len(lens)
print(f'\nBody length in a mid row group ({n:,} rows): '
      f'min={lens[0]}, median={lens[n//2]}, max={lens[-1]}, '
      f'empty={sum(1 for x in lens if x == 0):,}')
